In [1]:
import torch
import pandas as pd
import numpy as np
def cov(thetas, sigmas, N, ATT):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}

In [2]:
Ns = [500, 1000, 2000]
folder = "results_dgp_new_coef"
models = ['logistic', 'truncated_logistic', 'truncated_adv']
rows = []
methods =    [
        (1, "OLS"),
        (2, "Linear_old"),
        (3, "LASSO "),
        (4, "RF")]
for n in Ns:
    for model in models:
        file_suffix = f"N:{n}_{model}"
        pred_theta = torch.load(f"{folder}/{file_suffix}_pred_theta_OLS.pt")
        pred_sig = torch.load(f"{folder}/{file_suffix}_pred_sig_OLS.pt")
        ATT = torch.load(f"{folder}/{file_suffix}_ATT_calculations.pt")

        for idx, method_name in methods:
            est = pred_theta[:, idx-1]
            sig = pred_sig[:, idx-1]

            bias = (est - ATT).mean().item()#
            if method_name == "OLS":
                normalizer = 1
            else:
                normalizer = n
            rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
            cov_result = cov(est, sig,normalizer, ATT)
            coverage = cov_result["Coverage"].item()
            int_len = cov_result["interval_length"].item()

            rows.append([ method_name, bias, rmse, coverage, int_len])
        df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
        rows = []
        print(f"N:{n}, Model: {model}, ATT: {np.round(float(ATT),4)}")
        print(df.round(4).to_string(index=False))

N:500, Model: logistic, ATT: 5.1745
    Method    Bias   RMSE  Coverage  Interval Length
       OLS  0.0252 0.2685      0.99           1.1016
Linear_old -0.7938 0.8655      0.08           0.6063
    LASSO   0.5455 0.5995      0.61           1.2833
        RF -0.0159 0.2851      0.97           1.2516
N:500, Model: truncated_logistic, ATT: 4.478
    Method    Bias   RMSE  Coverage  Interval Length
       OLS -0.0268 0.2729      0.93           1.0298
Linear_old -0.4355 0.5159      0.33           0.6354
    LASSO   0.3251 0.4170      0.90           1.2362
        RF -0.0403 0.2769      0.97           1.2607
N:500, Model: truncated_adv, ATT: 4.5201
    Method    Bias   RMSE  Coverage  Interval Length
       OLS -0.0203 0.2692      0.95           1.0302
Linear_old -0.4565 0.5334      0.25           0.6354
    LASSO   0.3258 0.4133      0.91           1.2378
        RF -0.0333 0.2730      0.98           1.2612
N:1000, Model: logistic, ATT: 5.1744
    Method    Bias   RMSE  Coverage  Interval 

In [ ]:
df

,Method,Bias,RMSE,Coverage,Interval Length
0,OLS,-0.007795,0.123293,0.95,0.509559
1,Linear_old,-0.446236,0.464472,0.01,0.318078
2,LASSO,0.334247,0.359972,0.42,0.612839
3,RF,-0.013644,0.123169,0.98,0.627007


: 

testing

In [ ]:
# Parameters
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.estimateDiD_OLS import estimateDiD_OLS
from utils.estimateDiDLinear import estimateDiDLinear
import torch
import pandas as pd
import time
from torch.distributions import Normal

from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

class DiD_DGP:
    def __init__(self, dim_X=3, dim_Z = 3, beta_1=2.0, beta_2=1.5, c_1=1.0, delta_1 = 1 , delta_2 =1 , delta_3=1 , alpha_1 =1, gamma_1 = None, gamma_2= None, g  = None, gamma_10 = 0,gamma_11=0 ):
        self.dim_X = dim_X
        self.dim_Z = dim_Z
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.alpha_1 = alpha_1
        self.c_1 = c_1
        self.delta_1 = delta_1
        self.delta_2 = delta_2
        self.delta_3 = delta_3
        self.gamma_1 = gamma_1 # dimension dim_Z
        self.gamma_2 = gamma_2 # dimension dim_Z
        self.gamma_10 = gamma_10
        self.gamma_11 = gamma_11
        if gamma_1 is None: # if not specified, Z does not affect the propensity score
            self.gamma_1 = torch.zeros(dim_Z)
        if gamma_2 is None: # if not specified, Z does not affect the outcome
            self.gamma_2 = torch.zeros(dim_Z)
        self.g = g # this function specifies the propensity model
        self.ATT = None
       

    def generate(self, n, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        X1 = torch.randn(n, self.dim_X)
        X11 = X1[:, 0]
        eta = torch.randn(n)
        eps_x = torch.randn(n, self.dim_X)
        Z = torch.randn(n, self.dim_Z)
        Y1 = self.beta_1 * (X11 > 0).float() + self.delta_1 * X11 + Z @ self.gamma_2 + torch.randn(n) 
        prob_D = self.g(self.delta_2* X11 + Z @ self.gamma_1 + self.alpha_1 * Y1 + eta) # propensity score
        D = torch.bernoulli(prob_D)
        I_x11 = (X11 > 0)
        X2 = ((D*self.gamma_11 + (1-D)*self.gamma_10)*I_x11).unsqueeze(1)  +X1 + torch.ones(n, self.dim_X) *  (D *(1 + eta)).unsqueeze(1)  + eps_x
        X21 = X2[:, 0]
        Y2 = Y1 + self.c_1 * D.squeeze() + self.beta_2 * (X21 > 0).float() + self.delta_3* D.squeeze() * X21 + torch.randn(n)
        return {
            "X1": X1,
            "X2": X2,
            "Y1": Y1,
            "Y2": Y2,
            "D": D,
            "Z": Z,
        }    
    def simulate_ATT(self, n = 1000000):
        X1 = torch.randn(n, self.dim_X)
        X11 = X1[:, 0]
        I_x11 = (X11 > 0)

        eta = torch.randn(n)
        eps_x = torch.randn(n, self.dim_X)
        Z = torch.randn(n, self.dim_Z)
        Y1 = self.beta_1 * (X11 > 0).float() + self.delta_1 * X11 + Z @ self.gamma_2 + torch.randn(n) 
        prob_D = self.g(self.delta_2* X11 + Z @ self.gamma_1 + self.alpha_1 * Y1 + eta) 
        D = torch.bernoulli(prob_D)
        X2_0 =  (self.gamma_10*I_x11).unsqueeze(1) + X1 + eps_x 
        X2_1 =  (self.gamma_11*I_x11).unsqueeze(1)  + X1 + torch.ones(n, self.dim_X) *  (1 + eta).unsqueeze(1)  + eps_x
        X21_0 = X2_0[:, 0]
        X21_1 = X2_1[:, 0]
        # Calculate ATT
        E_X1D1 = torch.mean(X11[D == 1])  # E[X11 | D = 1]
        E_ETAD1 = torch.mean(eta[D == 1]) # E[ETA | D = 1]
        I_X21D1 = (X21_1 >0).float()
        I_X20D1 = (X21_0 > 0).float()
        E_X21D1 = torch.mean(I_X21D1[D == 1]) # P(X21(1) > 0 | D = 1)
        E_X20D1 = torch.mean(I_X20D1[D == 1])  # P(X21(0) > 0 | D = 1)
        E_X1D1_P = torch.mean(I_x11[D == 1].float()) # P(X11>0| D =1 )
        ATT = self.c_1 + self.beta_2 * E_X21D1 - self.beta_2 * E_X20D1 + self.delta_3 * E_X1D1 + self.delta_3* E_ETAD1 + self.delta_3*1 +self.delta_3*self.gamma_11*E_X1D1_P
        print(ATT)

        # sanity check
        Y_2_0 = Y1 + self.beta_2*I_X20D1
        Y_2_1 = Y1 + self.c_1 + self.beta_2* I_X21D1 + self.delta_3*X21_1
        print("Alternative Calc: ",  torch.mean((Y_2_1-Y_2_0)[D ==1]))
        del X1, X11, eta, eps_x, Z, Y1, prob_D, D, X2_0, X2_1, X21_0, X21_1, I_X21D1, I_X20D1

        return {"ATT": ATT, 
                "E_X1D1": E_X1D1, "E_ETAD1": E_ETAD1, 
                "E_X21D1": E_X21D1, "E_X20D1": E_X20D1,}
                
                

Ns = [500, 1000, 2000]
tmax = 100
dimX = 3
dimZ = 2
seed = 123 # this seed is for the DGP
folds = 5
# Bounds (only for truncated distributions)
lower = 0.30
upper = 0.70


lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : folds,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : 1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : 1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}






def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))


def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)


def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)


propensity_models = {
    'logistic': logistic,
    'truncated_logistic': truncated_logistic,
    'truncated_adv': truncated_adv
}

dgp = DiD_DGP(dim_X=dimX, dim_Z=dimZ, 
                        alpha_1 = 1, 
                    gamma_1=torch.ones(dimZ), 
                        gamma_2=torch.ones(dimZ),
                        g = logistic,
                        beta_2=2,
                        
                        delta_3= 2 ,
                        gamma_10=.5, gamma_11=1
                        )

data = dgp.generate(n=N*tmax, seed=seed)
X1, X2 = data['X1'], data['X2']
Y1, Y2 = data['Y1'], data['Y2']
Z, D = data['Z'], data['D']

ATT_calculations = dgp.simulate_ATT(n=100000000)


pred_theta = torch.zeros(tmax,3)
pred_sig = torch.zeros(tmax, 3)


In [ ]:
for t in tqdm(range(1)):
    torch.manual_seed(t)
    X1_sub = X1[t*N:(t+1)*N, :]
    X2_sub = X2[t*N:(t+1)*N, :]
    Y1_sub = Y1[t*N:(t+1)*N].view(-1, 1)
    Y2_sub = Y2[t*N:(t+1)*N].view(-1, 1)
    D_sub = D[t*N:(t+1)*N].view(-1, 1)
    Z_sub = Z[t*N:(t+1)*N, :]

    pred_theta[t,0], pred_sig[t,0] = estimateDiD_OLS(Y1_sub, Y2_sub, D_sub, Z_sub, X1_sub, X2_sub, seed= t)
    pred_theta[t,1], pred_sig[t,1],*_= estimateDiDLinear(Y1_sub, Y2_sub, D_sub, Z_sub, X1_sub, X2_sub, seed= t)
    pred_theta[t,2], pred_sig[t,2],*_ = estimateDynamicRiesz( Y1_sub, Y2_sub, D_sub, Z_sub, X1_sub, X2_sub, folds,
                                                                    method_a = "LASSO", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "LASSO", lasso_f_settings = lasso_f_settings,
                                                                        seed = t)


100%|██████████| 1/1 [00:06<00:00,  6.31s/it]


In [ ]:
import torch

class DiD_DGP:
    def __init__(
        self, dim_X=3, dim_Z=3,
        beta_1=2.0, beta_2=1.5, c_1=1.0,
        delta_1=1.0, delta_2=1.0, delta_3=1.0,
        alpha_1=1.0,
        gamma_1=None,  # dim_Z, effect of Z on propensity
        gamma_2=None,  # dim_Z, effect of Z on Y
        g=None,        # propensity link, maps R -> (0,1), e.g. torch.sigmoid
        gamma_10=0.0, gamma_11=0.0
    ):
        self.dim_X = dim_X
        self.dim_Z = dim_Z
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.c_1 = c_1
        self.delta_1 = delta_1
        self.delta_2 = delta_2
        self.delta_3 = delta_3
        self.alpha_1 = alpha_1
        self.gamma_1 = torch.zeros(dim_Z) if gamma_1 is None else gamma_1
        self.gamma_2 = torch.zeros(dim_Z) if gamma_2 is None else gamma_2
        self.gamma_10 = gamma_10
        self.gamma_11 = gamma_11
        self.g = (lambda s: torch.sigmoid(s)) if g is None else g  # default: logistic
        self.ATT = None

    @torch.no_grad()
    def generate(self, n, seed=None):
        if seed is not None:
            torch.manual_seed(seed)

        # Exogenous shocks / covariates
        X1 = torch.randn(n, self.dim_X)
        X11 = X1[:, 0]
        Z  = torch.randn(n, self.dim_Z)
        eta = torch.randn(n)                         # enters selection & X2 shift
        eps_x = torch.randn(n, self.dim_X)           # innovation for X2
        eps_y1 = torch.randn(n)                      # innovation for Y1
        eps_y2 = torch.randn(n)                      # innovation for Y2 (reused for both potentials!)

        # Period 1 outcome (pre)
        Y1 = self.beta_1 * (X11 > 0).float() + self.delta_1 * X11 + Z @ self.gamma_2 + eps_y1

        # Propensity & treatment draw (selection allowed to depend on Y1)
        score = self.delta_2 * X11 + Z @ self.gamma_1 + self.alpha_1 * Y1 + eta
        p = self.g(score).clamp(1e-6, 1-1e-6)        # probabilities in (0,1)
        D = torch.bernoulli(p)

        # Period 2: build both *potential* covariate paths X2(d) using same shocks
        I_x11 = (X11 > 0).float()
        ones = torch.ones(n, self.dim_X)

        def X2_given(d):
            # structural equation with do(D=d)
            shift = (d * (1 + eta)).unsqueeze(1)     # endogenous shock channel via eta
            gate  = ((d * self.gamma_11 + (1 - d) * self.gamma_10) * I_x11).unsqueeze(1)
            return gate + X1 + ones * shift + eps_x

        X2_0 = X2_given(0.0)
        X2_1 = X2_given(1.0)
        X21_0 = X2_0[:, 0]
        X21_1 = X2_1[:, 0]

        # Period 2: potential outcomes with same eps_y2
        Y2_0 = Y1 + 0.0 * self.c_1 + self.beta_2 * (X21_0 > 0).float() + 0.0 * self.delta_3 * X21_0 + eps_y2
        Y2_1 = Y1 + 1.0 * self.c_1 + self.beta_2 * (X21_1 > 0).float() + 1.0 * self.delta_3 * X21_1 + eps_y2

        # Factual post outcome comes from realized D
        Y2 = D * Y2_1 + (1 - D) * Y2_0
        X2 = D.unsqueeze(1) * X2_1 + (1 - D).unsqueeze(1) * X2_0

        # Ground-truth ATT: E[Y2(1) - Y2(0) | D = 1] under the realized selection
        treated = (D == 1)
        att_true = (Y2_1 - Y2_0)[treated].mean().item() if treated.any() else float('nan')
        self.ATT = att_true

        return {
            "X1": X1, "X2": X2,
            "Y1": Y1, "Y2": Y2,
            "Y2_0": Y2_0, "Y2_1": Y2_1,
            "D": D, "p": p, "Z": Z,
            "ATT_true": att_true
        }
    
    def simulate_att_population(self, n=int(5e6), batch_size=int(5e5), seed=None):
        """
        Approximate the population ATT = E[Y2(1) - Y2(0) | D=1]
        by simulating a very large sample in memory-efficient batches.

        Args:
            n (int): total number of simulated units
            batch_size (int): number of units per batch (controls memory use)
            seed (int or None): seed for reproducibility

        Returns:
            att (float): simulated ATT (large-n/population equivalent)
            share_treated (float): simulated P(D=1)
            n_treated (int): number of treated draws across all batches
        """
        if seed is not None:
            torch.manual_seed(seed)

        # Accumulators
        diff_sum_treated = 0.0
        n_treated = 0

        # Precompute constants
        ones_template = None  # set lazily to match dims once

        # How many full batches + remainder
        num_full = n // batch_size
        remainder = n - num_full * batch_size

        def run_batch(m):
            nonlocal diff_sum_treated, n_treated, ones_template

            # === Draw exogenous components
            X1 = torch.randn(m, self.dim_X)
            X11 = X1[:, 0]
            Z  = torch.randn(m, self.dim_Z)
            eta = torch.randn(m)
            eps_x  = torch.randn(m, self.dim_X)
            eps_y1 = torch.randn(m)
            eps_y2 = torch.randn(m)   # reused for both potentials

            # === Pre outcome
            Y1 = self.beta_1 * (X11 > 0).float() + self.delta_1 * X11 + Z @ self.gamma_2 + eps_y1

            # === Selection
            score = self.delta_2 * X11 + Z @ self.gamma_1 + self.alpha_1 * Y1 + eta
            p = self.g(score)
            D = torch.bernoulli(p)

            # === Build X2(d) under do(D=d) with same shocks
            I_x11 = (X11 > 0).float()
            if ones_template is None:
                ones_template = torch.ones(1, self.dim_X)

            def X2_given(d):
                d = float(d)
                shift = (d * (1 + eta)).unsqueeze(1)
                gate  = ((d * self.gamma_11 + (1 - d) * self.gamma_10) * I_x11).unsqueeze(1)
                return gate + X1 + ones_template.expand(m, -1) * shift + eps_x

            X2_0 = X2_given(0.0)
            X2_1 = X2_given(1.0)
            X21_0 = X2_0[:, 0]
            X21_1 = X2_1[:, 0]

            # === Potential outcomes with identical eps_y2
            Y2_0 = Y1 + self.beta_2 * (X21_0 > 0).float()  + eps_y2
            Y2_1 = Y1 + self.c_1 + self.beta_2 * (X21_1 > 0).float() + self.delta_3 * 1.0 * X21_1 + eps_y2

            # === ATT accumulator
            treated = (D == 1)
            if treated.any():
                diff_sum_treated += (Y2_1 - Y2_0)[treated].sum().item()
                n_treated += treated.sum().item()

        # Run full batches
        for _ in tqdm(range(num_full), desc = "Running diferent batches to simualte the true ATT"):
            run_batch(batch_size)

        # Run remainder if needed
        if remainder > 0:
            run_batch(remainder)

        # Compute ATT
        if n_treated == 0:
            att = float('nan')
        else:
            att = diff_sum_treated / n_treated

        # Store and return
        self.ATT = att
        share_treated = n_treated / n if n > 0 else float('nan')
        return att, share_treated, n_treated

In [ ]:

def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))


def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)


def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)


In [ ]:
Ns = [500, 1000, 2000]
tmax = 100
dimX = 3
dimZ = 2
seed = 123 # this seed is for the DGP
folds = 5
# Bounds (only for truncated distributions)
lower = 0.30
upper = 0.70
dgp = DiD_DGP(dim_X=dimX, dim_Z=dimZ, 
                        alpha_1 = 1, 
                    gamma_1=torch.ones(dimZ), 
                        gamma_2=torch.ones(dimZ),
                        g = truncated_adv,
                        beta_2=2,
                        
                        delta_3= 2 ,
                        gamma_10=.5, gamma_11=1
                        )
att_pop, share_treated, n_treated = dgp.simulate_att_population(
    n=5000000000, batch_size=50000, seed=123
)

Running diferent batches to simualte the true ATT: 100%|██████████| 100000/100000 [09:00<00:00, 185.14it/s]


In [ ]:
att_pop

5.117322375944214

In [ ]:
torch.load(f"results_new_dgp/N:2000_truncated_adv_ATT_calculations.pt")["ATT"]

tensor(5.1167)

## new ols

In [ ]:
import torch 
torch.load(f"results_new_ols/N:500_logistic_ATT_calculations_v2.pt")

4.548277116087663

In [ ]:
Ns = [500, 1000, 2000]
folder = "results_new_ols"
models = ['logistic', 'truncated_logistic', 'truncated_adv']
rows = []
methods =    [
        (1, "OLS")]
for n in Ns:
    for model in models:
        file_suffix = f"N:{n}_{model}"
        pred_theta = torch.load(f"{folder}/{file_suffix}_pred_theta_OLSv2.pt")
        pred_sig = torch.load(f"{folder}/{file_suffix}_pred_sig_OLSv2.pt")
        ATT = torch.load(f"{folder}/{file_suffix}_ATT_calculations_v2.pt")

        for idx, method_name in methods:
            est = pred_theta[:, idx-1]
            sig = pred_sig[:, idx-1]

            bias = (est - ATT).mean().item()#
            if method_name == "OLS":
                normalizer = 1
            else:
                normalizer = n
            rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
            cov_result = cov(est, sig,normalizer, ATT)
            coverage = cov_result["Coverage"].item()
            int_len = cov_result["interval_length"].item()

            rows.append([ method_name, bias, rmse, coverage, int_len])
        df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
        rows = []
        print(f"N:{n}, Model: {model}")
        print(df.round(4).to_string(index=False))

N:500, Model: logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0731 0.3044      0.96           1.1765
N:500, Model: truncated_logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS -0.016 0.2699      0.93           1.0242
N:500, Model: truncated_adv
Method    Bias   RMSE  Coverage  Interval Length
   OLS -0.0079 0.2686      0.94           1.0259
N:1000, Model: logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0696 0.2208      0.92           0.8257
N:1000, Model: truncated_logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0119 0.1822      0.93           0.7199
N:1000, Model: truncated_adv
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0229 0.1824      0.94           0.7192
N:2000, Model: logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0796 0.1701      0.93           0.5865
N:2000, Model: truncated_logistic
Method   Bias   RMSE  Coverage  Interval Length
   OLS 0.0015 0.1165      0.96           0.5071
N